In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json

import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils
import torch
import transformers
from rich.console import Console
from rich.theme import Theme
import builtins
_orig_print = builtins.print


custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme)


# Override print globally
def print(*args, **kwargs):
    console.print(*args, **kwargs)


logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

2026-03-05 12:38:39 __main__ INFO     torch.__version__='2.10.0+cu128', torch.version.cuda='12.8'
2026-03-05 12:38:40 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=8, torch.cuda.get_device_name()='NVIDIA A100 80GB PCIe'
2026-03-05 12:38:40 __main__ INFO     transformers.__version__='5.3.0'


## Loading the LM

In [3]:
from src.models import ModelandTokenizer

# model_key = "allenai/Olmo-3-1125-32B"
# model_key = "allenai/Olmo-3.1-32B-Instruct"
# model_key = "allenai/Olmo-3.1-7B-RL-Zero-Code"

# model_key = "meta-llama/Llama-3.3-70B-Instruct"
# model_key = "deepseek-ai/DeepSeek-R1-Distill-Llama-70B"
# model_key = "facebook/cwm"

# model_key = "deepseek-ai/deepseek-coder-33b-base"
# model_key = "deepseek-ai/deepseek-coder-33b-instruct"
# model_key = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"
# model_key = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"


# model_key = "openai/gpt-oss-20b"
# model_key = "openai/gpt-oss-120b" #! weird results. known issue.

# model_key = "Qwen/Qwen3-Coder-30B-A3B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-14B"
# model_key = "Qwen/Qwen2.5-Coder-14B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-32B"
# model_key = "Qwen/Qwen2.5-Coder-32B-Instruct"
# model_key = "Qwen/Qwen2.5-72B"
# model_key = "Qwen/Qwen2.5-72B-Instruct"
# model_key = "Qwen/Qwen3-Coder-Next"
# model_key = "Qwen/Qwen3-Coder-30B-A3B-Instruct"
model_key = "Qwen/Qwen3.5-27B"

# model_key = "bigcode/starcoder2-15b"

# model_key = "zai-org/GLM-4.7-Flash"

# model_key = "mistralai/Codestral-22B-v0.1"

mt = ModelandTokenizer(
    model_key=model_key,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

2026-03-05 12:38:44 src.models WARNING  Qwen/Qwen3.5-27B not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-03-05 12:38:44 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-03-05 12:38:44 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f529af2f020>
2026-03-05 12:38:44 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f529b684850> server_hostname='huggingface.co' timeout=10
2026-03-05 12:38:44 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f529b1b3e60>
2026-03-05 12:38:44 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-03-05 12:38:44 httpcore.http11 DEBUG    send_request_headers.complete
2026-03-05 12:38:44 httpcore.http11 DEB

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

2026-03-05 12:39:05 httpcore.connection DEBUG    close.started
2026-03-05 12:39:05 httpcore.connection DEBUG    close.complete
2026-03-05 12:39:05 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-03-05 12:39:05 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f529b3c7bc0>
2026-03-05 12:39:05 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f529b684850> server_hostname='huggingface.co' timeout=10
2026-03-05 12:39:05 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f529af52210>
2026-03-05 12:39:05 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-03-05 12:39:05 httpcore.http11 DEBUG    send_request_headers.complete
2026-03-05 12:39:05 httpcore.http11 DEBUG    send_request_body.started request=<Request [b'HE

### Debugging

In [4]:
# from nnsight import LanguageModel
# from transformers import Qwen3_5ForCausalLM

# lm = LanguageModel(
#     model_key,
#     dispatch=True,
#     cache_dir=env_utils.HF_CACHE_DIR,
# )
# lm

In [5]:
# type(lm._model)

In [6]:
from src.functional import predict_next_token
prompt = "Eiffel Tower is locate in the city of"

predict_next_token(
    mt=mt,
    inputs=prompt
)

[[PredictedToken(token=' Paris', prob=0.25390625, logit=14.8125, token_id=11751, metadata=None),
  PredictedToken(token='\n', prob=0.1279296875, logit=14.125, token_id=198, metadata=None),
  PredictedToken(token='\n\n', prob=0.08251953125, logit=13.6875, token_id=271, metadata=None),
  PredictedToken(token=' __', prob=0.07275390625, logit=13.5625, token_id=1259, metadata=None),
  PredictedToken(token='...', prob=0.056640625, logit=13.3125, token_id=1076, metadata=None)]]

In [7]:
# lm.config

In [8]:
# from datasets import load_dataset

# ds = load_dataset("gonglinyuan/safim", "block")

In [9]:
# sample = ds["test"][16]
# sample

In [10]:
# print(sample["prompt"])

In [11]:
# print(sample["eval_prompt"])

In [12]:
# instruction = """Instruction: Complete the {{completion}} in the following Python function.\n\n"""
# code = """# the function calculates (a ^ b) mod m for large positive integers a, b, and m.
# def big_mod_exp(a: int, b: int, m: int) -> int:
#     result = 1
#     a = a % m
#     while b > 0:
#         if {{completion}}:
#             result = (result * a) % m
#         b = b >> 1
#         a = (a * a) % m
#     return result

# Answer: 0 =="""

# instruction = """Complete the following Python function.\n\n"""
# code = """Calculate (a ^ b) mod m for large positive integers a, b, and m.
# def big_mod_exp(a: int, b: int, m: int) -> int:
#     result = 1
#     a = a % m
#     while b > 0:
#         if 0 =="""

## Evaluation

In [13]:
from src.goals.eval import run_evaluation, pretty_print_eval_results
sample_idx = 5


In [14]:
from src.goals.data import IncompleteCodeDataset

eval_dataset = IncompleteCodeDataset.load_dataset()
print(eval_dataset[sample_idx])

2026-03-05 12:39:13 src.goals.data INFO     Loaded 17 samples from /disk/u/arnab/Codes/Projects/goals/data/incomplete_code.json


IncompleteCodeSample(
    instruction='Write a function in python that checks if a string is a palindrome.\n\n',
    code='def is_palindrome(s: str) -> bool:\n    return True if ',
    unit_tests=[
        ('racecar', True),
        ('hello', False),
        ('', True),
        ('a', True),
        ('ab', False),
        ('aba', True),
        ('abba', True)
    ],
    prefills={
        'aligned': ['', '(', 's', 's ==', 's == s', 's[::', 's[::-1]', '(s', 'list(', 'list(s)'],
        'adversarial': ['not', 'not (', 's +', '3', '1 +'],
        'inconsequential': ['bool(', 'True and', 'False or', '0 +']
    },
    function_name='is_palindrome'
)

In [15]:
from src.goals.functional import apply_chat_template

sample = eval_dataset[sample_idx]
prompt = apply_chat_template(
    mt=mt,
    user_query=sample.instruction.strip(),
    incomplete_code=sample.code,
)

# prompt = sample["instruction"] + "<code>" + sample["code"]
_orig_print(f'[white]"{prompt}"[/white]')

[white]"<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a function in python that checks if a string is a palindrome.<|im_end|>
<|im_start|>assistant
<think>

</think>

<code>
def is_palindrome(s: str) -> bool:
    return True if"[/white]


In [16]:
from src.functional import generate_with_patch, predict_next_token

output = generate_with_patch(
    mt=mt,
    inputs=prompt,
    n_gen_per_prompt=1,
    max_new_tokens=64,
    do_sample=False,
    remove_prefix=True,
    skip_special_tokens=False,
)

print(f'[white]"{output[0]}"[/white]')

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


" s == s[::-1] else False
</code><|im_end|>"

In [17]:
# from anthropic import Anthropic

# # MODEL_NAME = "claude-haiku-4-5-20251001"
# # MODEL_NAME = "claude-sonnet-4-5-20250929"
# MODEL_NAME = "claude-opus-4-5-20251101"
# client = Anthropic(api_key=env_utils.load_env_var("CLAUDE_KEY"))

# response = client.messages.create(
#     model=MODEL_NAME,
#     max_tokens=200,
#     temperature=0.6,
#     system="You are a helpful assistant proficient in programming.",
#     messages=[
#         {
#             "role": "user",
#             "content": "Write a function in python that checks if a number is odd.",
#         },
#         {
#             "role": "assistant",
#             "content": """def is_odd(n: int) -> bool:
#     return True if (1 +""",
#         }
#     ],
# )

# print(response.content[0].text)

In [18]:
# mt.tokenizer.special_tokens_map

In [19]:
mt.name.lower()

'qwen/qwen3.5-27b'

In [22]:
from src.tokens import prepare_input
from src.goals.data import CodeGenerations, Code
from src.goals.functional import supports_chat_template, apply_chat_template

sample = eval_dataset[sample_idx]
instruction = sample.instruction
unfinished_code = sample.code
function_name = sample.function_name or "func"
unit_tests = sample.unit_tests
prefills = []
for ptype, pfill_list in sample.prefills.items():
    prefills.extend([prefill, ptype] for prefill in pfill_list)


if supports_chat_template(mt):
    prompts = [
        apply_chat_template(
            mt=mt,
            user_query=instruction.strip(),
            incomplete_code=unfinished_code + prefill,
        )
        for prefill, ptype in prefills
    ]
else:
    prompts = [
        instruction + "<code>" + unfinished_code + prefill
        for (prefill, ptype) in prefills
    ]


inputs = prepare_input(
    prompts=prompts,
    tokenizer=mt,
)

output = generate_with_patch(
    mt=mt,
    inputs=inputs,
    n_gen_per_prompt=1,
    max_new_tokens=64,
    do_sample=False,
    remove_prefix=True,
)

results = CodeGenerations(
    sample=sample,
    generations=[
        Code(prefill=prefill, prefill_type=ptype, full_prompt=inp, lm_generation=out)
        for (prefill, ptype), inp, out in zip(prefills, prompts, output)
    ],
)

# print(f"[white]{str(results)}[/white]")

eval_results = run_evaluation(
    result = results,
    function_name=function_name,
    code_stop_tag="</code>",
)

# Print results in tabular format
pretty_print_eval_results(eval_results)

"""def is_palindrome(s: str) -> bool:
    return True if """

                                                           Code Completion Results                                                            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┓
┃ Prefill                     ┃ Logic                                       ┃ Verdict ┃ Unit Tests                                    ┃ Logs ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━┩
│ aligned: ""                 │ " s == s[::-1] else False"                  │ PASSED  │                                               │      │
├─────────────────────────────┼─────────────────────────────────────────────┼─────────┼───────────────────────────────────────────────┼──────┤
│ aligned: "("                │ "(s == s[::-1]) else False"                 │ PASSED  │                                               │      │
├─────────────────────────────┼─────────────────────────────────────────────┼─────────┼───────────────────────────────────────────────┼──────┤
│ aligned: "s"                │ "s == s[::-1] else False"                   │ PASSED  │                                               │      │
├─────────────────────────────┼─────────────────────────────────────────────┼─────────┼───────────────────────────────────────────────┼──────┤
│ aligned: "s =="             │ "s == s[::-1] else False"                   │ PASSED  │                                               │      │
├─────────────────────────────┼─────────────────────────────────────────────┼─────────┼───────────────────────────────────────────────┼──────┤
│ aligned: "s == s"           │ "s == s[::-1] else False"                   │ PASSED  │                                               │      │
├─────────────────────────────┼─────────────────────────────────────────────┼─────────┼───────────────────────────────────────────────┼──────┤
│ aligned: "s[::"             │ "s[::1] == s[::-1] else False"              │ PASSED  │                                               │      │
├─────────────────────────────┼─────────────────────────────────────────────┼─────────┼───────────────────────────────────────────────┼──────┤
│ aligned: "s[::-1]"          │ "s[::-1] == s else False"                   │ PASSED  │                                               │      │
├─────────────────────────────┼─────────────────────────────────────────────┼─────────┼───────────────────────────────────────────────┼──────┤
│ aligned: "(s"               │ "(s == s[::-1]) else False"                 │ PASSED  │                                               │      │
├─────────────────────────────┼─────────────────────────────────────────────┼─────────┼───────────────────────────────────────────────┼──────┤
│ aligned: "list("            │ "list(                                      │ PASSED  │                                               │      │
│                             │         s.lower()                           │         │                                               │      │
│                             │     ) == list(                              │         │                                               │      │
│                             │         s.lower()                           │         │                                               │      │
│                             │     )[::-1] else False"                     │         │                                               │      │
├─────────────────────────────┼─────────────────────────────────────────────┼─────────┼───────────────────────────────────────────────┼──────┤
│ aligned: "list(s)"          │ "list(s) == list(s)[::-1] else False"       │ PASSED  │                                               │      │
├─────────────────────────────┼─────────────────────────────────────────────┼─────────┼───────────────────────────────────────────────┼────

Summary: 15/19 passed, 4/19 failed

In [23]:
out_idx = -2
code = eval_results.generations[out_idx]
print(code)

print(f'Full Prompt:\n[white]"{code.full_prompt}"[/white]\n')
print(f'Full Output:\n[white]"{code.lm_generation}"[/white]\n')

Code(
    prefill='False or',
    prefill_type='inconsequential',
    full_prompt='<|im_start|>system\nYou are a helpful assistant proficient in programming. When completing code, 
ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> 
tags.<|im_end|>\n<|im_start|>user\nWrite a function in python that checks if a string is a 
palindrome.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n<code>\ndef is_palindrome(s: str) -> bool:\n  
return True if False or',
    lm_generation=' (s == s[::-1]) else False',
    source_code='def is_palindrome(s: str) -> bool:\n    return True if False or (s == s[::-1]) else False',
    function_name='is_palindrome',
    eval=SingleCodeEval(
        code='def is_palindrome(s: str) -> bool:\n    return True if False or (s == s[::-1]) else False',
        verdict='PASSED',
        unit_test_results=[
            UnitTestResult(input='racecar', expected=True, actual='True', passed=True, error=None),
            UnitTestResult(input='hello', expected=False, actual='False', passed=True, error=None),
            UnitTestResult(input='', expected=True, actual='True', passed=True, error=None),
            UnitTestResult(input='a', expected=True, actual='True', passed=True, error=None),
            UnitTestResult(input='ab', expected=False, actual='False', passed=True, error=None),
            UnitTestResult(input='aba', expected=True, actual='True', passed=True, error=None),
            UnitTestResult(input='abba', expected=True, actual='True', passed=True, error=None)
        ],
        warnings=[],
        error_message=None,
        prefill=('False or', 'inconsequential'),
        output=' (s == s[::-1]) else False'
    )
)

Full Prompt:
"<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a function in python that checks if a string is a palindrome.<|im_end|>
<|im_start|>assistant
<think>

</think>

<code>
def is_palindrome(s: str) -> bool:
    return True if False or"

Full Output:
" (s == s[::-1]) else False"

In [24]:
# with open("test.json", "w") as f:
#     json.dump(eval_results.to_dict(), f, indent=4)

# with open("test.json", "r") as f:
#     test_gen = json.load(f)

# test_gen = CodeGenerations.from_dict(test_gen)
# pretty_print_eval_results(eval_results, output_file="print_test.txt")

In [25]:
# prompt = prompts[-1]
# print(prompt)

# predict_next_token(mt=mt, inputs=prompt, k=20)

In [26]:
from tqdm.auto import tqdm
from src.goals.functional import supports_chat_template

save_dir = os.path.join(env_utils.DEFAULT_RESULTS_DIR, model_key.split("/")[-1])
os.makedirs(save_dir, exist_ok=True)

for sample_idx, sample in enumerate(tqdm(eval_dataset)):
    instruction = sample.instruction
    unfinished_code = sample.code
    function_name = sample.function_name or "func"
    unit_tests = sample.unit_tests
    prefills = []
    for ptype, pfill_list in sample.prefills.items():
        prefills.extend([prefill, ptype] for prefill in pfill_list)

    if supports_chat_template(mt):
        prompts = [
            apply_chat_template(
                mt=mt,
                user_query=instruction.strip(),
                incomplete_code=unfinished_code + prefill,
            )
            for prefill, ptype in prefills
        ]
    else:
        prompts = [
            instruction + "<code>" + unfinished_code + prefill
            for (prefill, ptype) in prefills
        ]

    inputs = prepare_input(
        prompts=prompts,
        tokenizer=mt,
    )

    output = generate_with_patch(
        mt=mt,
        inputs=inputs,
        n_gen_per_prompt=1,
        max_new_tokens=64,
        do_sample=False,
        remove_prefix=True,
    )

    code_completions = CodeGenerations(
        sample=sample,
        generations=[
            Code(
                prefill=prefill, prefill_type=ptype, full_prompt=inp, lm_generation=out
            )
            for (prefill, ptype), inp, out in zip(prefills, prompts, output)
        ],
    )

    # Run evaluation (returns JSON-like dict)
    code_completions = run_evaluation(
        result=code_completions,
        function_name=function_name,
        code_stop_tag="</code>",
    )

    # Save results to JSON file
    save_path = os.path.join(save_dir, f"eval_result_{sample_idx}.json")
    with open(save_path, "w") as f:
        json.dump(code_completions.to_dict(), f, indent=4)
        
    pretty_print_eval_results(
        code_completions,
        output_file=os.path.join(save_dir, f"pretty_print_{sample_idx}.txt"),
    )

    logger.info(f"Saved evaluation results to {save_path}")

  0%|          | 0/17 [00:00<?, ?it/s]

"""def is_odd(n: int) -> bool:
    return True if """

                                                    Code Completion Results                                                    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┓
┃ Prefill                     ┃ Logic                                    ┃ Verdict ┃ Unit Tests                        ┃ Logs ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━┩
│ aligned: ""                 │ " n % 2 != 0 else False                  │ PASSED  │                                   │      │
│                             │                                          │         │                                   │      │
│                             │ # Example usage:                         │         │                                   │      │
│                             │ # print(is_odd(3))  # Output: True       │         │                                   │      │
│                             │ # print(is_odd(4))  # Output: False"     │         │                                   │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "("                │ "(n % 2) != 0 else False"                │ PASSED  │                                   │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "1"                │ "1 else False"                           │ FAILED  │ is_odd(1)=True ✓                  │      │
│                             │                                          │         │ is_odd(2)=True, expected False ✗  │      │
│                             │                                          │         │ is_odd(0)=True, expected False ✗  │      │
│                             │                                          │         │ is_odd(15)=True ✓                 │      │
│                             │                                          │         │ is_odd(-3)=True ✓                 │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "1 =="             │ "1 == n % 2 else False"                  │ PASSED  │                                   │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "0"                │ "0 < n % 2 else False"                   │ PASSED  │                                   │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "(1"               │ "(1 & n) else False"                     │ PASSED  │                                   │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "(0"               │ "(0 if n % 2 == 0 else 1) else False"    │ PASSED  │                                   │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "abs("             │ "abs(remainder := n % 2) == 1 else False │ PASSED  │                                   │      │
│                             │                                          │         │                                   │      │
│                             │ # Example usage:                         │         │                                   │      │
│                             │ # print(is_odd(3))   # True              │         │                                   │      │
│                             │ # print(is_odd(4))   # False"            │         │                                   │      │
├─────────────────────────────┼─

Summary: 15/20 passed, 5/20 failed

2026-03-05 12:42:43 __main__ INFO     Saved evaluation results to /disk/u/arnab/Codes/Projects/goals/results/Qwen3.5-27B/eval_result_0.json


"""def is_even(n: int) -> bool:
    return True if """

... (59 outputs left out)

"""def has_unique_elements(lst: list) -> bool:
    return True if """

                                                                                        Code Completion Results                                                                                         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Prefill                     ┃ Logic                                               ┃ Verdict      ┃ Unit Tests                                          ┃ Logs                                        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ aligned: ""                 │ " len(lst) == len(set(lst)) else False              │ PASSED       │                                                     │                                             │
│                             │                                                     │              │                                                     │                                             │
│                             │ # Example usage:                                    │              │                                                     │                                             │
│                             │ # print(has_unique_elements([1, 2, 3, 4]))      #   │              │                                                     │                                             │
│                             │ True                                                │              │                                                     │                                             │
│                             │ # print(has_unique_elements([1, 2, 2, 3]))      #   │              │                                                     │                                             │
│                             │ False"                                              │              │                                                     │                                             │
├─────────────────────────────┼─────────────────────────────────────────────────────┼──────────────┼─────────────────────────────────────────────────────┼─────────────────────────────────────────────┤
│ aligned: "("                │ "(len(lst) == len(set(lst))) else False"            │ PASSED       │                                                     │                                             │
├─────────────────────────────┼─────────────────────────────────────────────────────┼──────────────┼─────────────────────────────────────────────────────┼─────────────────────────────────────────────┤
│ aligned: "len("             │ "len("                                              │ SYNTAX_ERROR │                                                     │ '(' was never closed (<string>, line 2)     │
│                             │                                                     │              │                                                     │ def has_unique_elements(lst: list) -> bool: │
│                             │                                                     │              │                                                     │     return True if len(                     │
├─────────────────────────────┼─────────────────────────────────────────────────────┼──────────────┼─────────────────────────────────────────────────────┼─────────────────────────────────────────────┤
│ aligned: "len(lst)"         │ "len(lst) == len(set(lst)) else False               │ PASSED       │                                                     │                                             │
│                             │                                                     │              │                                                     │                           

Summary: 10/17 passed, 7/17 failed

2026-03-05 12:47:54 __main__ INFO     Saved evaluation results to /disk/u/arnab/Codes/Projects/goals/results/Qwen3.5-27B/eval_result_16.json


In [26]:
# with open(os.path.join(save_dir, "eval_result_3.json"), "r") as f:
#     eval_demo = json.load(f)

# pretty_print_eval_results(eval_demo, output_file="results.txt")